# C03. Качество потока кадров
Авторский синтетический эксперимент. Единица наблюдения — устройство за день. `expected` — ожидаемые кадры, `received` — доставленные, `good` — годные среди доставленных. Время — условные дни 1 и 2. Реальных данных нет.

Запуск: установите зависимости из `requirements-projects.txt`, откройте notebook, выполните **Restart + Run All**. Доставка = сумма received / сумма expected. Качество = сумма good / сумма received только для строк с известным good; охват показываем отдельно. На нулевом знаменателе — NaN. Это два разных вопроса.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print({'python':sys.version.split()[0], 'numpy':np.__version__, 'pandas':pd.__version__})
raw = pd.DataFrame([
    ('a',1,900,900,810), ('b',1,100,50,25),
    ('a',2,100,100,90), ('b',2,900,450,225),
    ('c',1,100,50,np.nan), ('c',2,100,0,np.nan)
], columns=['device','day','expected','received','good'])
raw

In [ ]:
def analyze(raw):
    df=raw.copy()
    assert not df.duplicated(['device','day']).any()
    assert (df['expected']>0).all()
    assert ((df['received']>=0)&(df['received']<=df['expected'])).all()
    known=df['good'].notna()
    assert ((df.loc[known,'good']>=0)&(df.loc[known,'good']<=df.loc[known,'received'])).all()
    df['delivery']=df['received']/df['expected']
    df['quality']=df['good']/df['received'].replace(0,np.nan)
    summaries=[]
    for day,g in df.groupby('day'):
        k=g['good'].notna()
        denominator=g.loc[k,'received'].sum()
        received=g['received'].sum()
        summaries.append(dict(day=day,delivery=received/g['expected'].sum(),
            quality=g.loc[k,'good'].sum()/denominator if denominator else np.nan,
            quality_coverage=denominator/received if received else np.nan))
    return df,pd.DataFrame(summaries).set_index('day')
rows,summary=analyze(raw)
assert raw.shape==(6,5)
assert rows['quality'].isna().sum()==2
pd.testing.assert_frame_equal(summary,analyze(raw)[1])
summary

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4),sharey=True)
for device,g in rows.groupby('device'):
    axes[0].plot(g['day'],g['delivery'],marker='o',label=device)
    axes[1].plot(g['day'],g['quality'],marker='o',label=device)
for ax,title in zip(axes,['Доставка / ожидаемые кадры','Годные / полученные кадры']):
    ax.set(title=title,xlabel='День',ylabel='Доля',ylim=(0,1.05),xticks=[1,2])
    ax.legend(title='Устройство');ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

In [ ]:
for day,row in summary.iterrows():
    print(f"День {day}: доставка {row.delivery:.1%}; качество {row.quality:.1%}; охват качества {row.quality_coverage:.1%}")
# У a и b показатели не изменились. Меняется доля их кадров.
for device in ['a','b']:
    g=rows[rows.device==device]
    assert g.delivery.nunique()==1 and g.quality.nunique()==1

## Вывод и самостоятельная часть
В синтетических данных показатели a и b постоянны, но общий результат меняется вместе с составом. У c качество неизвестно: нулевая доставка во второй день не даёт оценки качества полученных кадров. Это наблюдение о данном наборе, не доказательство причины изменения реального потока.

1. Измените ожидаемые объёмы a и b, сохраняя их доли доставки и качества. Повторите все ячейки.
2. Добавьте пропуск good у a: как меняются охват и смысл общего качества?
3. Сохраните HTML-отчёт и объясните числители, знаменатели и ограничения. Проверьте, что все числа текста получены из summary, а не переписаны вручную.